# UNSW-NB15 Label Harmonization  
*@Ullas*

Builds a deterministic `attack_cat → int` mapping that every downstream module
(multiclass training, inference, API) will use.  
Output: `models/unsw_label_mapping.json`

In [ ]:
import os
import glob
import json
import pandas as pd

# Resolve repo root regardless of working directory
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DATA_DIR   = os.path.join(REPO_ROOT, "data", "UNSW-NB15")
MODELS_DIR = os.path.join(REPO_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")

## 1 — Load Training Data

Prefer `UNSW_NB15_training-set.csv` if present; otherwise concatenate the four
raw partitions `UNSW-NB15_1.csv … UNSW-NB15_4.csv`.  
We only need the `attack_cat` and `label` columns for this notebook.

In [ ]:
# UNSW-NB15 CSV column names (the raw files have no header row)
UNSW_COLS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "sload", "dload", "spkts", "dpkts", "swin", "dwin", "stcpb", "dtcpb",
    "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "sjit", "djit",
    "stime", "ltime", "sintpkt", "dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "ct_src_ltm", "attack_cat", "label"
]

# Try training-set CSV first, then fall back to raw partitions
training_set_path = os.path.join(DATA_DIR, "UNSW_NB15_training-set.csv")

if os.path.exists(training_set_path):
    print("Loading UNSW_NB15_training-set.csv …")
    df = pd.read_csv(training_set_path)
else:
    # Raw partition files (no header)
    raw_paths = sorted(glob.glob(os.path.join(DATA_DIR, "UNSW-NB15_*.csv")))
    if not raw_paths:
        raise FileNotFoundError(
            f"No UNSW-NB15 CSV files found in {DATA_DIR}. "
            "Download them from https://research.unsw.edu.au/projects/unsw-nb15-dataset"
        )
    print(f"Loading {len(raw_paths)} raw CSV partition(s) …")
    parts = [pd.read_csv(p, header=None, names=UNSW_COLS, low_memory=False) for p in raw_paths]
    df = pd.concat(parts, ignore_index=True)

print(f"\nShape          : {df.shape}")
print(f"Columns present: {list(df.columns[:10])} …")
print(f"\nLabel dist:\n{df['label'].value_counts().to_string()}")
df[["attack_cat", "label"]].head(10)

## 2 — Explore Attack Categories

Check unique values, value counts, nulls, and whitespace inconsistencies.

In [ ]:
# Normalize: strip whitespace, unify casing (title-case)
df["attack_cat"] = df["attack_cat"].astype(str).str.strip()
# Replace empty / nan strings that slipped in as text
df["attack_cat"] = df["attack_cat"].replace({"nan": "Normal", "": "Normal"})

null_count = df["attack_cat"].isnull().sum()
print(f"Null attack_cat values : {null_count}")

vc = df["attack_cat"].value_counts()
print(f"\nattack_cat value counts ({len(vc)} unique):\n{vc.to_string()}")

print(f"\nUnique attack_cat values (raw):\n{sorted(df['attack_cat'].unique())}")

## 3 — Build Deterministic Integer Mapping

Rules:
- `Normal` → **0** (background traffic, must be class 0)
- All attack types sorted **alphabetically** → 1, 2, 3 …

This is deterministic: anyone running this notebook on any UNSW-NB15 subset
will always produce the same mapping.

In [ ]:
unique_cats = df["attack_cat"].unique().tolist()

# Separate Normal; sort the rest alphabetically
attack_types = sorted([c for c in unique_cats if c != "Normal"])

# str → int  (Normal=0, attacks=1…N)
str_to_int: dict[str, int] = {"Normal": 0}
for idx, cat in enumerate(attack_types, start=1):
    str_to_int[cat] = idx

# Inverse: int → str  (used by multiclass_inference.py)
int_to_str: dict[int, str] = {v: k for k, v in str_to_int.items()}

print("String → Integer mapping:")
for cat, idx in sorted(str_to_int.items(), key=lambda x: x[1]):
    print(f"  {idx:2d}  {cat}")

print(f"\nTotal classes: {len(str_to_int)}")

## 4 — Apply Mapping to DataFrame

Add the `attack_cat_id` integer column and spot-check both columns side by side.

In [ ]:
df["attack_cat_id"] = df["attack_cat"].map(str_to_int)

# Spot-check
sample = df[["attack_cat", "attack_cat_id", "label"]].drop_duplicates("attack_cat").sort_values("attack_cat_id")
print("Per-category sample:")
print(sample.to_string(index=False))

print(f"\nNull attack_cat_id rows: {df['attack_cat_id'].isnull().sum()}")

## 5 — Save Label Mapping to JSON

`unsw_label_mapping.json` contains both directions:
- `"str_to_int"` — used during training to assign `attack_cat_id`  
- `"int_to_str"` — used at inference time by `multiclass_inference.py` to decode predictions

In [ ]:
mapping_payload = {
    "str_to_int": str_to_int,
    # JSON keys must be strings; callers cast to int when loading
    "int_to_str": {str(k): v for k, v in int_to_str.items()},
    "num_classes": len(str_to_int),
    "notes": (
        "Normal=0; attack types sorted alphabetically assign 1..N. "
        "Generated by label_harmonization.ipynb. Do not edit manually."
    ),
}

output_path = os.path.join(MODELS_DIR, "unsw_label_mapping.json")
with open(output_path, "w") as f:
    json.dump(mapping_payload, f, indent=2)

print(f"Saved → {output_path}")

# Quick round-trip check
with open(output_path) as f:
    loaded = json.load(f)
assert loaded["str_to_int"] == str_to_int, "Round-trip mismatch!"
print("Round-trip verification passed ✓")

## 6 — Verify Mapping Correctness

Assertions every reviewer should see pass before the PR is opened.

In [ ]:
import sys
sys.path.insert(0, REPO_ROOT)

# Expected values hardcoded in config/config.py
EXPECTED_MAPPING = {
    "Normal": 0,
    "Analysis": 1,
    "Backdoors": 2,
    "DoS": 3,
    "Exploits": 4,
    "Fuzzers": 5,
    "Generic": 6,
    "Reconnaissance": 7,
    "Shellcode": 8,
    "Worms": 9,
}

# --- assertion 1: no nulls in attack_cat_id ---
assert df["attack_cat_id"].isnull().sum() == 0, "FAIL: attack_cat_id has nulls"
print("✓ No null attack_cat_id values")

# --- assertion 2: Normal → 0 ---
assert str_to_int.get("Normal") == 0, "FAIL: Normal must map to 0"
print("✓ Normal → 0")

# --- assertion 3: 10 total classes (Normal + 9 attack types) ---
assert len(str_to_int) == 10, f"FAIL: expected 10 classes, got {len(str_to_int)}"
print(f"✓ Total classes = {len(str_to_int)}")

# --- assertion 4: mapping matches config.py ---
for cat, expected_id in EXPECTED_MAPPING.items():
    actual_id = str_to_int.get(cat)
    assert actual_id == expected_id, (
        f"FAIL: '{cat}' → {actual_id}, expected {expected_id}. "
        "Update UNSW_ATTACK_TYPE_MAPPING in config/config.py to match."
    )
print("✓ Mapping matches config/config.py UNSW_ATTACK_TYPE_MAPPING")

# --- summary table ---
print("\n── Final Mapping ─────────────────────────")
print(f"{'ID':>4}  {'Category':<20}  {'Count':>8}")
print("─" * 38)
for cat_id in sorted(int_to_str):
    cat_name = int_to_str[cat_id]
    count = int((df["attack_cat"] == cat_name).sum())
    print(f"{cat_id:>4}  {cat_name:<20}  {count:>8,}")
print("─" * 38)
print(f"{'':>4}  {'TOTAL':<20}  {len(df):>8,}")
print("\n✅ All assertions passed — unsw_label_mapping.json is ready.")